# TP — Voyage impossible (impossible travel)

Objectif : détecter les **déplacements impossibles** — un utilisateur qui se
connecte depuis deux lieux trop éloignés en trop peu de temps (vitesse
supérieure à **900 km/h**, la vitesse d'un avion de ligne) — puis éliminer les
faux positifs pour ne garder que le compte réellement compromis.

**Consignes :** complétez chaque fonction (corps entre `#BEGIN` / `#END`), puis
exécutez la cellule de tests associée.

**Cadre légal — données simulées :** ce jeu de données est **entièrement
simulé** (adresses IP réservées à la documentation, RFC 5737). Toute analyse
de journaux d'authentification réels exige une **autorisation écrite**.
**Art. 323-1 Code pénal :** accès frauduleux = délit, même « juste pour voir ».


## 0. Les données

Le SIEM vous remet un journal de connexions **minimal** : ni géolocalisation,
ni information sur les VPN. Ces informations se **reconstruisent** en joignant
deux autres fichiers, comme avec une vraie base GeoIP.

| fichier | colonnes | rôle |
| --- | --- | --- |
| `connexions.csv` | `timestamp`, `username`, `ip` | le journal brut des connexions |
| `ip_geo.csv` | `ip`, `latitude`, `longitude`, `city`, `country` | géolocalisation de chaque IP |
| `vpn_ips.csv` | `ip` | IP de sortie VPN/datacenter connues |

**Aide pandas :**
- `pd.read_csv(..., parse_dates=["timestamp"])` : charge et convertit la colonne en dates.
- `.head()`, `.info()` : premier aperçu.
- Doc : https://pandas.pydata.org/docs/user_guide/10min.html


In [ ]:
import pandas as pd
import numpy as np

connexions = pd.read_csv("connexions.csv", parse_dates=["timestamp"])
geo = pd.read_csv("ip_geo.csv")
vpn = pd.read_csv("vpn_ips.csv")

print(connexions.shape, geo.shape, vpn.shape)
connexions.head()



In [ ]:
connexions.info()
geo.head()



## Exercice 1 — Reconstruire la géolocalisation

`connexions.csv` ne contient que l'IP. Pour calculer des distances, il faut
ajouter la latitude/longitude de chaque IP via une **jointure** (`merge`) sur
la colonne commune `ip`.

**À propos de `merge` :** `pd.merge(gauche, droite, on="ip", how="left")`
joint deux DataFrames sur une colonne commune. `how="left"` garde toutes les
lignes de gauche, même si aucune géolocalisation n'existe (la latitude serait
alors `NaN`).
Doc : https://pandas.pydata.org/docs/reference/api/pandas.merge.html


In [ ]:
def fusionner(connexions: pd.DataFrame, geo: pd.DataFrame) -> pd.DataFrame:
    """Joint les connexions à la géolocalisation de leur IP.

    Colonnes ajoutées : latitude, longitude, city, country.
    """
    # TODO
    pass



In [ ]:
df = fusionner(connexions, geo)
assert "latitude" in df.columns and "longitude" in df.columns
assert not df["latitude"].isna().any(), "des IP n'ont pas de géolocalisation"
print("ex 1 OK :", df.shape[0], "lignes,", df["ip"].nunique(), "IP distinctes")



## Exercice 2 — Détection naïve : tri, précédent, vitesse

Pour détecter un déplacement impossible, on compare chaque connexion d'un
utilisateur à sa connexion **précédente** (même idée que `get_diffs` du TD
beacon, qui comparait chaque instant au précédent) :

1. **Trier** par `username` puis `timestamp` — sans ce tri, on comparerait
   deux connexions d'utilisateurs différents.
2. **Parcourir** les lignes en retenant la précédente (« shift à la main ») :
   si elle appartient au **même utilisateur**, on calcule la distance et la
   durée entre les deux ; sinon on l'ignore.
3. Calculer la **vitesse** = distance (haversine) / durée.
4. Alerter si la vitesse dépasse 900 km/h.

La distance du grand cercle est fournie ci-dessous (vous n'avez pas à la
réécrire) :



In [ ]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distance du grand cercle (km) entre deux points (lat/lon en degrés)."""
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))



In [ ]:
def detecter_voyages_impossibles(df: pd.DataFrame, seuil_kmh: float = 900) -> pd.DataFrame:
    """Renvoie les connexions impliquant un déplacement > seuil_kmh.

    Colonnes retournées : username, timestamp, ip, t_prev, ip_prev,
    distance_km, duree_h, vitesse_kmh (une ligne par alerte).
    """
    # TODO
    # Indice : trier par ["username", "timestamp"], puis parcourir les lignes
    # en retenant la précédente (comme get_diffs du TD beacon).
    # Indice : ne comparer que si la précédente est du même utilisateur.
    # Indice : distance avec haversine_km, durée en heures avec .total_seconds() / 3600.
    pass



In [ ]:
alertes = detecter_voyages_impossibles(df)
print(len(alertes), "alertes,", alertes["username"].nunique(), "utilisateurs distincts")

assert len(alertes) == 120, f"attendu 120 alertes, obtenu {len(alertes)}"
assert alertes["username"].nunique() == 18
print("ex 2 OK : 120 alertes… mais la plupart sont des faux positifs !")



In [ ]:
alertes

Le détecteur remonte **beaucoup** d'alertes. La plupart sont des **faux
positifs** : la position d'une IP n'est **pas** la position de l'utilisateur.

**Deux sources de faux positifs :**
1. **Le VPN** : un utilisateur chez lui se connecte via un nœud de sortie VPN
   à l'étranger → sa position semble « sauter » de sa ville vers le nœud.
2. **L'IP d'entreprise** : des employés sortent par un **même** proxy/NAT,
   géolocalisé au siège → chacun semble « sauter » de son domicile vers le
   siège.

On va les éliminer l'un après l'autre.


## Exercice 3 — Éliminer les VPN (la liste)

Les nœuds de sortie VPN sont **connus** et listés dans `vpn_ips.csv`. On filtre
les connexions dont l'IP figure dans cette liste.

**Pourquoi une liste suffit ici :** un nœud de sortie VPN peut n'être utilisé
que par **un seul** de nos utilisateurs : il n'est pas « partagé », donc
indétectable par une simple statistique. Seule une **liste externe** (ou une
base d'ASN de datacenters) l'attrape.


In [ ]:
def retirer_vpn(df: pd.DataFrame, vpn_ips: pd.DataFrame) -> pd.DataFrame:
    """Renvoie les connexions dont l'IP n'est PAS une sortie VPN connue."""
    # TODO
    pass



In [ ]:
sans_vpn = retirer_vpn(df, vpn)
alertes_vpn = detecter_voyages_impossibles(sans_vpn)
print(len(alertes_vpn), "alertes,", alertes_vpn["username"].nunique(), "utilisateurs")

assert len(alertes_vpn) == 85
assert alertes_vpn["username"].nunique() == 13
print("ex 3 OK : les télétravailleurs VPN ne sont plus alertés")



## Exercice 4 — Éliminer l'IP d'entreprise (l'heuristique)

L'IP de sortie de l'entreprise n'est dans **aucune** liste. Mais elle est
**partagée** : de nombreux utilisateurs distincts se connectent derrière elle.

**Heuristique : une IP partagée = infrastructure.** `df.groupby("ip")["username"].nunique()`
compte les utilisateurs distincts par IP. Une IP résidentielle n'est vue que
par 1 (ou très peu) d'utilisateurs ; un proxy/NAT/VPN est vu par **beaucoup**.

En vrai, on fixe un seuil : un IP partagé par une poignée de membres d'une
même famille reste légitime. Ici, l'IP d'entreprise est partagée par
12 personnes, une IP résidentielle par 1.


In [ ]:
def ip_partagees(df: pd.DataFrame, seuil: int = 2) -> set[str]:
    """IP vues par au moins `seuil` utilisateurs distincts (infra partagée)."""
    # TODO
    pass



In [ ]:
partagees = ip_partagees(sans_vpn)
assert partagees == {"203.0.113.10"}, f"IP partagées : {partagees}"
nb = sans_vpn[sans_vpn["ip"] == "203.0.113.10"]["username"].nunique()
print("ex 4 OK :", partagees, "partagé par", nb, "utilisateurs")



## Exercice 5 — Synthèse : qui est réellement compromis ?

On applique les deux filtres successivement, puis on relance la détection. Ce
qui reste n'est ni un VPN, ni une IP partagée : c'est un déplacement
impossible **réel** — un **compte compromis** (*account takeover*).

**Les deux filtres sont complémentaires :** le VPN mono-utilisateur échappe à
l'heuristique (ex 4) mais pas à la liste (ex 3). L'IP d'entreprise échappe à
la liste mais pas à l'heuristique. Il faut **les deux**.


In [ ]:
def utilisateurs_compromis(df: pd.DataFrame, vpn_ips: pd.DataFrame) -> list[str]:
    """Utilisateurs au déplacement impossible réel, VPN et IP partagée écartés."""
    # TODO
    pass



In [ ]:
compromis = utilisateurs_compromis(df, vpn)
assert compromis == ["alice.dupont"], f"trouvé : {compromis}"
print("ex 5 OK : le compte compromis est", compromis[0])

# Regardons de plus près son déplacement :
alerte = detecter_voyages_impossibles(df[df["username"] == "alice.dupont"])
print(alerte[["timestamp", "ip", "t_prev", "ip_prev",
             "distance_km", "vitesse_kmh"]].to_string(index=False))



## Bilan

| exercice | faux positif éliminé | méthode | notion pandas |
| --- | --- | --- | --- |
| 2 | (aucun — détection brute) | seuil de vitesse > 900 km/h | `sort_values` + boucle sur le précédent |
| 3 | VPN (5 télétravailleurs) | liste externe `vpn_ips.csv` | `isin` |
| 4 | IP d'entreprise (12 employés) | IP partagée par N utilisateurs | `groupby().nunique()` |
| 5 | — | il ne reste que `alice.dupont` | enchaînement des filtres |

**À retenir :**

- La **position d'une IP n'est pas la position de l'utilisateur** (VPN, NAT,
  proxy) : une géolocalisation se lit avec prudence.
- La détection d'*impossible travel* se **croise toujours** avec d'autres
  signaux (liste de VPN, IP partagées) pour écarter les faux positifs.
- Chaque faux positif éliminé rapproche du vrai signal.

